# Phase 6: Model Optimization, Class Imbalance, and Threshold Analysis

This phase tunes Logistic Regression, Random Forest, Gradient Boosting, and XGBoost using five-fold `StratifiedKFold` on the training partition. Every search estimator contains the preprocessing `ColumnTransformer`, so each fold fits its own preprocessing. F1 is the refit/optimization metric; accuracy, precision, recall, and ROC-AUC are also reported.

A stratified validation slice is reserved from the existing training set for model and threshold selection. The holdout test set is passed only to the final evaluation stage. Standard and imbalance-weighted search settings are compared; SMOTE is omitted because `imbalanced-learn` is not installed and class weighting provides a suitable in-scope comparison.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

working_directory = Path.cwd()
project_root = working_directory if (working_directory / 'src').is_dir() else working_directory.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.model_training import create_train_test_split, RANDOM_STATE
from src.model_optimization import run_phase6

data_path = project_root / 'data' / 'processed' / 'telco_churn_cleaned.csv'
df = pd.read_csv(data_path)
X_train, X_test, y_train, y_test = create_train_test_split(
    df, target_col='Churn', test_size=0.2, random_state=RANDOM_STATE
)
print(f'Training rows: {len(X_train):,}; held-out test rows: {len(X_test):,}')
print(f'Train churn rate: {y_train.mean():.3%}; test churn rate: {y_test.mean():.3%}')

Training rows: 5,634; held-out test rows: 1,409
Train churn rate: 26.535%; test churn rate: 26.544%


## Validation-Based Optimization

The optimizer further splits only `X_train`/`y_train` into model-fitting and validation portions. Grid search and threshold selection do not receive `X_test` or `y_test`. The selected pipeline is then refit on all training rows before the single final test evaluation.

In [2]:
results = run_phase6(
    X_train,
    y_train,
    X_test,
    y_test,
    artifact_path=str(project_root / 'models' / 'final_churn_model.joblib'),
    reports_dir=str(project_root / 'reports'),
    figures_dir=str(project_root / 'reports' / 'figures'),
    cv_splits=5,
    random_state=RANDOM_STATE,
)
print('Selected model:', results['selected_model'])
print('Validation-selected threshold:', results['selected_threshold'])

Selected model: Gradient Boosting
Validation-selected threshold: 0.35


## Validation Results

CV columns summarize the best F1-scoring grid configuration under five-fold stratified CV. Validation columns use the threshold selected only from internal validation probabilities. These are selection results, not final test metrics.

In [3]:
display(results['validation_comparison'])
display(results['threshold_analysis'])

,Model,Best Parameters,CV Accuracy,CV Precision,CV Recall,CV F1,CV ROC-AUC,Validation Threshold,Validation Accuracy,Validation Precision,Validation Recall,Validation F1,Validation ROC-AUC
0,Gradient Boosting,"{'model__learning_rate': 0.05, 'model__max_dep...",0.800978,0.660511,0.515115,0.577786,0.846816,0.35,0.801242,0.606232,0.715719,0.656442,0.851437
1,XGBoost,"{'model__learning_rate': 0.05, 'model__max_dep...",0.748170,0.516267,0.810237,0.630579,0.843284,0.65,0.817214,0.659794,0.642140,0.650847,0.849920
2,Random Forest,"{'model__class_weight': 'balanced', 'model__ma...",0.775013,0.562087,0.699052,0.622421,0.841216,0.50,0.788820,0.583106,0.715719,0.642643,0.844207
3,Logistic Regression,"{'model__C': 1.0, 'model__class_weight': 'bala...",0.747284,0.515392,0.801862,0.627448,0.843288,0.65,0.801242,0.616099,0.665552,0.639871,0.848335


,Threshold,Precision,Recall,F1,Predicted Positive Rate,TP,FP,TN,FN
0,0.10,0.400287,0.933110,0.560241,0.618456,279,418,410,20
1,0.15,0.445000,0.892977,0.593993,0.532387,267,333,495,32
2,0.20,0.483431,0.829431,0.610837,0.455191,248,265,563,51
3,0.25,0.513978,0.799331,0.625654,0.412600,239,226,602,60
4,0.30,0.556650,0.755853,0.641135,0.360248,226,180,648,73
5,0.35,0.606232,0.715719,0.656442,0.313221,214,139,689,85
6,0.40,0.638710,0.662207,0.650246,0.275067,198,112,716,101
7,0.45,0.671815,0.581940,0.623656,0.229814,174,85,743,125
8,0.50,0.704762,0.494983,0.581532,0.186335,148,62,766,151
9,0.55,0.744444,0.448161,0.559499,0.159716,134,46,782,165


## FINAL UNTOUCHED TEST EVALUATION

The test set is used below only after model and threshold selection are complete. The saved artifact includes the fitted preprocessing/model pipeline and selected threshold. Note that this same holdout was reported for baseline evaluation in earlier phases; it is held out from Phase 6 optimization, but is not globally unseen within the project history.

In [4]:
final_test_results = pd.DataFrame([results['final_test_metrics']])
display(final_test_results)
print('Artifact:', results['artifact_path'])
print('Figures saved under:', project_root / 'reports' / 'figures')

,Accuracy,Precision,Recall,F1,ROC-AUC,TP,FP,TN,FN
0,0.781405,0.569915,0.719251,0.635934,0.845443,269,203,832,105


Artifact: C:\Users\aasye\Desktop\churn-prediction\churn-prediction\models\final_churn_model.joblib
Figures saved under: C:\Users\aasye\Desktop\churn-prediction\churn-prediction\reports\figures


## Limitations

Threshold and candidate selection can overfit the internal validation slice; the final holdout provides a confirmation, not proof of future performance. The original holdout has already appeared in baseline reports. Class weighting is a modeling comparison, not a substitute for estimating operational intervention costs. This analysis establishes associations and predictive performance, not causal effects or guaranteed churn reduction.